In [8]:
#r "nuget: Plotly.NET, 5.0.0"
#r "nuget: Plotly.NET.Interactive, 5.0.0"
#r "nuget: Plotly.NET.CSharp, 0.13.0"
#r "nuget: MathNet.Numerics, 5.0.0"

using System;
using System.Collections.Generic;
using System.IO;
using System.Linq;
using System.Globalization;
using Plotly.NET;
using Plotly.NET.LayoutObjects;
using Plotly.NET.Interactive;
using Plotly.NET.CSharp;
using Chart = Plotly.NET.CSharp.Chart;
using MathNet.Numerics.LinearAlgebra;
using MathNet.Numerics.LinearAlgebra.Double;
using System.Text.RegularExpressions;

public class Track
{
    public string Genre { get; set; }
    public string ArtistName { get; set; }
    public string TrackName { get; set; }
    public string TrackId { get; set; }
    public double Popularity { get; set; }
    public double[] Features { get; set; } 
    public double[] NormalizedFeatures { get; set; } 
    public double[] ClusterFeatures { get; set; }
    public int Cluster { get; set; }
    public double[] PCA { get; set; } 
}

string filePath = "SpotifyFeatures.csv";
var lines = File.ReadAllLines(filePath).Skip(1).ToList();
var allTracks = new List<Track>();
var regex = new Regex(",(?=(?:[^\"]*\"[^\"]*\")*[^\"]*$)");

foreach (var line in lines)
{
    var cols = regex.Split(line);
    if (cols.Length < 18) continue;
    try
    {
        allTracks.Add(new Track
        {
            Genre = cols[0],
            ArtistName = cols[1],
            TrackName = cols[2],
            TrackId = cols[3],
            Popularity = double.Parse(cols[4], CultureInfo.InvariantCulture),
            Features = new double[]
            {
                double.Parse(cols[5], CultureInfo.InvariantCulture),  
                double.Parse(cols[6], CultureInfo.InvariantCulture),  
                double.Parse(cols[7], CultureInfo.InvariantCulture),  
                double.Parse(cols[8], CultureInfo.InvariantCulture),  
                double.Parse(cols[9], CultureInfo.InvariantCulture),  
                double.Parse(cols[11], CultureInfo.InvariantCulture), 
                double.Parse(cols[12], CultureInfo.InvariantCulture), 
                double.Parse(cols[14], CultureInfo.InvariantCulture), 
                double.Parse(cols[15], CultureInfo.InvariantCulture), 
                double.Parse(cols[17], CultureInfo.InvariantCulture)  
            }
        });
    }
    catch {}
}

var featureNames = new[] { "Acousticness", "Danceability", "Duration", "Energy", "Instrumentalness", "Liveness", "Loudness", "Speechiness", "Tempo", "Valence" };
int featureCount = featureNames.Length;
double[] mins = new double[featureCount];
double[] maxs = new double[featureCount];
foreach (var t in allTracks.Take(5))
{
    Console.WriteLine($"{t.ArtistName} - {t.TrackName} | Genre: {t.Genre} | Pop: {t.Popularity}");
}

for (int i = 0; i < featureCount; i++)
{
    mins[i] = allTracks.Min(t => t.Features[i]);
    maxs[i] = allTracks.Max(t => t.Features[i]);
    double avg = allTracks.Average(t => t.Features[i]);
    Console.WriteLine($"{featureNames[i],-16} | Min: {mins[i],10:F3} | Max: {maxs[i],10:F3} | Avg: {avg,10:F3}");
}

foreach (var track in allTracks)
{
    track.NormalizedFeatures = new double[featureCount];
    for (int i = 0; i < featureCount; i++)
    {
        double range = maxs[i] - mins[i];
        track.NormalizedFeatures[i] = range == 0 ? 0 : 2 * (track.Features[i] - mins[i]) / range - 1;
    }
    
    track.ClusterFeatures = track.NormalizedFeatures
        .Where((val, index) => index != 2) 
        .ToArray();
}
Console.WriteLine($"Треків: {allTracks.Count}");
var popularTracks = allTracks.Where(t => t.Popularity >= 85).ToList();
Console.WriteLine($"Залишилося треків >= 85: {popularTracks.Count}");

(int[] labels, double inertia) RunKMeans(List<double[]> data, int k, int maxIterations = 100)
{
    var rand = new Random(2411);
    int n = data.Count;
    int dims = data[0].Length;
    var centroids = new List<double[]>();
    for (int i = 0; i < k; i++) centroids.Add((double[])data[rand.Next(n)].Clone());
    
    int[] labels = new int[n];
    double inertia = 0;

    for (int iter = 0; iter < maxIterations; iter++)
    {
        bool changed = false;
        inertia = 0;
        for (int i = 0; i < n; i++)
        {
            double minDist = double.MaxValue;
            int bestCluster = 0;
            for (int c = 0; c < k; c++)
            {
                double dist = 0;
                for (int d = 0; d < dims; d++) dist += Math.Pow(data[i][d] - centroids[c][d], 2);
                if (dist < minDist) { minDist = dist; bestCluster = c; }
            }
            if (labels[i] != bestCluster) changed = true;
            labels[i] = bestCluster;
            inertia += minDist;
        }
        if (!changed) break;
        
        var newCentroids = new double[k][];
        int[] counts = new int[k];
        for (int c = 0; c < k; c++) newCentroids[c] = new double[dims];
        for (int i = 0; i < n; i++)
        {
            int cluster = labels[i];
            counts[cluster]++;
            for (int d = 0; d < dims; d++) newCentroids[cluster][d] += data[i][d];
        }
        for (int c = 0; c < k; c++)
        {
            if (counts[c] == 0) continue;
            for (int d = 0; d < dims; d++) centroids[c][d] = newCentroids[c][d] / counts[c];
        }
    }
    return (labels, inertia);
}

var dataForClustering = popularTracks.Select(t => t.ClusterFeatures).ToList();

var inertias = new List<double>();
var kValues = Enumerable.Range(2, 14).ToList();

foreach (int k in kValues)
{
    var result = RunKMeans(dataForClustering, k, 100);
    inertias.Add(result.inertia);
}

var elbowChart = Chart.Line<int, double, string>(x: kValues, y: inertias, Name: "Inertia")
    .WithTitle("Elbow Method")
    .WithXAxisStyle(title: Plotly.NET.Title.init("k"))
    .WithYAxisStyle(title: Plotly.NET.Title.init("Inertia"));
elbowChart.Display();


int optimalK = 5; 
var finalModel = RunKMeans(dataForClustering, optimalK, 100);

for (int i = 0; i < popularTracks.Count; i++)
{
    popularTracks[i].Cluster = finalModel.labels[i];
}

for (int c = 0; c < optimalK; c++)
{
    int count = popularTracks.Count(t => t.Cluster == c);
    Console.WriteLine($"Кластер {c}: {count} треків");
}



var matrix = Matrix<double>.Build.DenseOfRowArrays(popularTracks.Select(t => t.ClusterFeatures).ToArray());
var covMatrix = (matrix.Transpose() * matrix) / (matrix.RowCount - 1);
var evd = covMatrix.Evd();
var eigenVectors = evd.EigenVectors;
var pcaComponents = eigenVectors.SubMatrix(0, eigenVectors.RowCount, eigenVectors.ColumnCount - 3, 3);
var projectedData = matrix * pcaComponents;

for (int i = 0; i < popularTracks.Count; i++)
{
    popularTracks[i].PCA = projectedData.Row(i).ToArray();
}

var charts3D = new List<GenericChart>();
for (int c = 0; c < optimalK; c++)
{
    var clusterTracks = popularTracks.Where(t => t.Cluster == c).ToList();
    if(clusterTracks.Any())
    {
        charts3D.Add(Chart.Scatter3D<double, double, double, string>(
            x: clusterTracks.Select(t => t.PCA[2]), 
            y: clusterTracks.Select(t => t.PCA[1]), 
            z: clusterTracks.Select(t => t.PCA[0]),
            mode: StyleParam.Mode.Markers,
            MultiText: clusterTracks.Select(t => $"{t.ArtistName} - {t.TrackName}").ToArray(), 
            Name: $"Cluster {c}"
        ));
    }
}
Chart.Combine(charts3D).WithTitle("PCA 3D (Без Duration)").Display();



Console.Write($"{"Кластер",-8}");
foreach (var name in featureNames.Where(n => n != "Duration")) Console.Write($"{name.Substring(0, Math.Min(5, name.Length)),8}");
Console.WriteLine();

for (int c = 0; c < optimalK; c++)
{
    var tracksInCluster = popularTracks.Where(t => t.Cluster == c).ToList();
    if (!tracksInCluster.Any()) continue;
    
    Console.Write($"{c,-8}");
    foreach (int f in Enumerable.Range(0, featureCount).Where(i => i != 2)) 
    {
        double avg = tracksInCluster.Average(t => t.NormalizedFeatures[f]);
        Console.Write($"{avg,8:F2}");
    }
    Console.WriteLine();
}

for (int c = 0; c < optimalK; c++)
{
    var tracksInCluster = popularTracks.Where(t => t.Cluster == c).ToList();
    if (!tracksInCluster.Any()) continue; 
    
    var featureAverages = new List<(string Name, double Value)>();
    for (int f = 0; f < featureNames.Length; f++)
    {
        if (f == 2) continue; 
        double avg = tracksInCluster.Average(t => t.NormalizedFeatures[f]); 
        featureAverages.Add((featureNames[f], avg));
    }
    
    var topFeatures = featureAverages
        .OrderByDescending(f => f.Value)
        .Take(3)
        .Select(f => f.Name);
        
    Console.WriteLine($"Кластер {c}: ({string.Join(", ", topFeatures)}).");
    foreach (var t in tracksInCluster.Take(3))
    {
        Console.WriteLine($" - {t.ArtistName} : {t.TrackName}");
    }
    Console.WriteLine();
}

Installed Packages MathNet.Numerics, 5.0.0 Plotly.NET, 5.0.0 Plotly.NET.CSharp, 0.13.0 Plotly.NET.Interactive, 5.0.0

Henri Salvador - C'est beau de faire un Show | Genre: Movie | Pop: 0
Martin & les fées - Perdu d'avance (par Gad Elmaleh) | Genre: Movie | Pop: 1
Joseph Williams - Don't Let Me Be Lonely Tonight | Genre: Movie | Pop: 3
Henri Salvador - Dis-moi Monsieur Gordon Cooper | Genre: Movie | Pop: 0
Fabien Nataf - Ouverture | Genre: Movie | Pop: 4
Acousticness     | Min:      0.000 | Max:      0.996 | Avg:      0.369
Danceability     | Min:      0.057 | Max:      0.989 | Avg:      0.554
Duration         | Min:  15387.000 | Max: 5552917.000 | Avg: 235122.339
Energy           | Min:      0.000 | Max:      0.999 | Avg:      0.571
Instrumentalness | Min:      0.000 | Max:      0.999 | Avg:      0.148
Liveness         | Min:      0.010 | Max:      1.000 | Avg:      0.215
Loudness         | Min:    -52.457 | Max:      3.744 | Avg:     -9.570
Speechiness      | Min:      0.022 | Max:      0.967 | Avg:      0.121
Tempo            | Min:     30.379 | Max:    242.903 | Avg:    117.667
Valence          | M

<!-- Plotly chart will be drawn inside this DIV -->

Кластер 0: 65 треків
Кластер 1: 92 треків
Кластер 2: 106 треків
Кластер 3: 63 треків
Кластер 4: 91 треків


<!-- Plotly chart will be drawn inside this DIV -->

Кластер    Acous   Dance   Energ   Instr   Liven   Loudn   Speec   Tempo   Valen
0          -0.65    0.46    0.55   -1.00   -0.76    0.71   -0.80   -0.18    0.52
1          -0.71    0.32    0.32   -0.99   -0.66    0.67   -0.86   -0.06   -0.46
2          -0.85    0.37    0.42   -1.00   -0.65    0.68   -0.81   -0.03    0.07
3          -0.81    0.75    0.01   -0.99   -0.58    0.58   -0.54   -0.28   -0.15
4           0.07    0.37    0.02   -0.99   -0.72    0.61   -0.79   -0.28   -0.09
Кластер 0: (Loudness, Energy, Valence).
 - Jonas Brothers : Sucker
 - DJ Snake : "Taki Taki (with Selena Gomez, Ozuna & Cardi B)"
 - Hailee Steinfeld : "Let Me Go (with Alesso, Florida Georgia Line & watt)"

Кластер 1: (Loudness, Danceability, Energy).
 - Ariana Grande : "break up with your girlfriend, i'm bored"
 - Ariana Grande : "thank u, next"
 - Ariana Grande : in my head

Кластер 2: (Loudness, Energy, Danceability).
 - Ariana Grande : bloodline
 - Ariana Grande : bad idea
 - Lauv : i'm so tired...

Клас